[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/project-walkthrough/01_crawl_storage/01_crawl_storage_solutions.ipynb)

# 01. `crawl-storage-example` 동행 — 연습 문제 해설

> 본문: [01_crawl_storage.ipynb](01_crawl_storage.ipynb)

먼저 직접 풀어본 뒤에 보세요. 정답 코드보다 **왜 그렇게 하는지**가 중요합니다.

## 0. 환경 준비 — 프로젝트를 옆에 펼쳐두기

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
print("Colab에서 실행 중:", IN_COLAB)

if IN_COLAB:
    # 이 노트북은 "예제 프로젝트를 옆에 두고 같이 읽는" 노트북입니다.
    # 그래서 설명만 하지 않고, 저장소를 통째로 내려받아 **실제 프로젝트 파일**을 열어봅니다.
    subprocess.run(["git", "clone", "-q", "https://github.com/karzit/temp.git", "/content/temp"], check=False)
    REPO_ROOT = "/content/temp"
    !pip install -q requests beautifulsoup4 python-dotenv psycopg2-binary
else:
    # 로컬에서 열었다면 이 노트북 위치(notebooks/project-walkthrough/NN_xxx/)에서 3단계 위가 저장소 루트입니다.
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))

PROJECT = os.path.join(REPO_ROOT, "example-projects", "crawl-storage-example")
SRC = os.path.join(PROJECT, "src")
print("프로젝트 경로:", PROJECT)
assert os.path.isdir(SRC), "프로젝트 경로를 찾지 못했습니다. 저장소 루트에서 노트북을 열었는지 확인하세요."

아래 `show()`는 이 노트북 전체에서 쓰는 도우미입니다. **설명 대신 진짜 프로젝트 파일을 그대로 출력**해서, 노트북과 코드가 어긋나지 않게 합니다.

In [ ]:
import re


def show(filename, start=None, end=None, grep=None):
    """프로젝트 파일의 실제 소스를 줄 번호와 함께 출력한다.

    설명을 읽는 것과 실제 코드를 보는 것 사이의 간격을 없애기 위한 도우미입니다.
    이 노트북에서 "코드 읽기"라고 나오는 곳은 전부 진짜 프로젝트 파일을 그대로 보여줍니다.

        show("crawl.py")                  전체
        show("crawl.py", 30, 45)          30~45번째 줄
        show("crawl.py", grep="def ")     'def '가 들어간 줄만
    """
    path = os.path.join(SRC, filename) if not os.path.isabs(filename) else filename
    lines = open(path, encoding="utf-8").read().splitlines()

    if grep:
        picked = [(i, l) for i, l in enumerate(lines, 1) if re.search(grep, l)]
    else:
        s = (start or 1) - 1
        e = end or len(lines)
        picked = [(i, l) for i, l in enumerate(lines[s:e], s + 1)]

    for i, line in picked:
        print(f"{i:>4} | {line}")


def show_file(relpath, **kwargs):
    """프로젝트 루트 기준 경로로 파일을 보여준다 (README, docker-compose 등)."""
    show(os.path.join(PROJECT, relpath), **kwargs)


# 프로젝트 소스를 import할 수 있도록 경로를 등록해둡니다.
if SRC not in sys.path:
    sys.path.insert(0, SRC)

## 연습 1. `Content-Type`으로 PDF 판별하기

**문제**: `crawl_one()`은 URL이 `.pdf`로 끝나는지만 봅니다.
응답 헤더의 `Content-Type`도 보도록 고치세요.

**왜 필요한가**: 사내 시스템의 첨부파일 링크는 `/download?fileId=1234`처럼 확장자가 없는 경우가 흔합니다.
지금 코드는 이런 PDF를 **HTML로 착각해서 BeautifulSoup에 넣습니다.**
그러면 PDF 바이너리에서 억지로 뽑아낸 깨진 글자가 DB에 텍스트로 저장됩니다.
에러도 안 나기 때문에 나중에 검색이 이상해지고 나서야 발견됩니다.

In [ ]:
import requests
from bs4 import BeautifulSoup

from crawl import extract_text_from_html


def is_pdf(url: str, response: requests.Response) -> bool:
    """URL 확장자와 응답 헤더를 함께 보고 PDF인지 판별한다.

    둘 중 하나만 맞아도 PDF로 본다. 확장자는 있는데 헤더가 없는 서버도 있고,
    그 반대도 있기 때문이다. Content-Type에는 "application/pdf; charset=..."처럼
    뒤에 옵션이 붙어 오는 경우가 있어서 `in`으로 느슨하게 검사한다.
    """
    if url.lower().endswith(".pdf"):
        return True
    content_type = response.headers.get("Content-Type", "").lower()
    return "application/pdf" in content_type


def crawl_one_v2(url: str, save) -> None:
    """개선판 crawl_one. 저장 함수를 인자로 받아 테스트하기 쉽게 만들었다."""
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
    response.raise_for_status()

    if is_pdf(url, response):
        save(url=url, content_type="pdf", text_content=None, binary_content=response.content)
    else:
        save(url=url, content_type="html", text_content=extract_text_from_html(response.text), binary_content=None)


# 판별 로직만 따로 테스트해봅니다. 네트워크 없이 가짜 응답으로 확인합니다.
class FakeResponse:
    def __init__(self, content_type):
        self.headers = {"Content-Type": content_type}


cases = [
    ("https://ex.com/a.pdf", "text/html", "확장자만 PDF"),
    ("https://ex.com/download?id=1", "application/pdf; charset=binary", "헤더만 PDF"),
    ("https://ex.com/page", "text/html; charset=utf-8", "둘 다 아님"),
]
for url, ctype, label in cases:
    print(f"{label:<14} {url:<32} -> PDF? {is_pdf(url, FakeResponse(ctype))}")

> 💡 **더 나아가기**: 헤더도 확장자도 못 믿는 경우가 있습니다.
> PDF 파일은 항상 `%PDF-`라는 5바이트로 시작하므로, `response.content[:5] == b"%PDF-"`로
> **내용을 직접 확인**하는 게 가장 확실합니다. 이걸 매직 넘버(magic number) 검사라고 합니다.

## 연습 2. 크롤링 실패를 DB에 기록하기

**문제**: 지금은 실패하면 `print`만 하고 끝납니다. 어떤 URL이 언제 왜 실패했는지 남기려면?

**설계 판단이 먼저입니다.** 실패 기록을 `crawled_documents`에 같이 넣을지, 따로 뺄지.
- **같은 테이블**: `status` 컬럼 추가. 간단하지만 "성공한 문서만" 조회할 때 항상 조건을 붙여야 합니다.
- **별도 테이블**: 성공 데이터가 깨끗하게 유지됩니다. 같은 URL이 여러 번 실패한 이력도 쌓을 수 있습니다.

크롤링 실패는 **일회성 사건**이고 재시도 판단에 이력이 필요하므로, 여기서는 별도 테이블로 갑니다.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("""
CREATE TABLE crawl_failures (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    url TEXT NOT NULL,
    error_type TEXT NOT NULL,     -- 'timeout', 'http_404' 처럼 분류해두면 집계가 쉬워진다
    error_message TEXT,
    failed_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    retry_count INTEGER NOT NULL DEFAULT 0
)
""")
# url에 UNIQUE를 걸지 않은 것이 포인트다. 같은 URL이 여러 번 실패한 이력을 그대로 쌓아야
# "이 사이트는 계속 죽어 있다"와 "한 번 튕겼다"를 구분할 수 있다.


def record_failure(url, error):
    """예외 객체에서 분류와 메시지를 뽑아 기록한다."""
    error_type = type(error).__name__
    conn.execute(
        "INSERT INTO crawl_failures (url, error_type, error_message) VALUES (?, ?, ?)",
        (url, error_type, str(error)[:200]),
    )
    conn.commit()


record_failure("https://ex.com/a", requests.Timeout("연결 시간 초과"))
record_failure("https://ex.com/a", requests.Timeout("연결 시간 초과"))
record_failure("https://ex.com/b", requests.HTTPError("404 Not Found"))

print("실패 이력:")
for row in conn.execute("SELECT url, error_type, error_message FROM crawl_failures"):
    print("  ", row)

print("\nURL별 실패 횟수 (재시도 판단에 사용):")
for row in conn.execute("SELECT url, COUNT(*) FROM crawl_failures GROUP BY url ORDER BY COUNT(*) DESC"):
    print("  ", row)

**결과를 읽는 법**: `/a`가 2번 실패했습니다. 이런 집계가 있으면
"3회 이상 실패한 URL은 목록에서 제외" 같은 운영 규칙을 만들 수 있습니다.
`print`만 했다면 터미널을 닫는 순간 사라졌을 정보입니다.

## 연습 3. 변경 감지 — 해시로 실제 변경만 골라내기

**문제**: 내용이 **실제로 바뀐 경우에만** `crawled_at`을 갱신하려면?

**왜 필요한가**: 지금은 재크롤링할 때마다 `crawled_at`이 갱신됩니다.
그러면 "최근 일주일 안에 바뀐 규정"을 뽑을 수가 없습니다. 전부 오늘 날짜니까요.

내용이 같은지 비교하려면 본문 전체를 대조해야 하는데, 수십 KB 텍스트를 매번 비교하는 건 낭비입니다.
그래서 **해시**를 씁니다. 긴 텍스트를 짧은 지문 하나로 줄여서 그것만 비교하는 거죠.

In [ ]:
import hashlib

conn2 = sqlite3.connect(":memory:")
conn2.execute("""
CREATE TABLE crawled_documents (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    url TEXT NOT NULL UNIQUE,
    text_content TEXT,
    content_hash TEXT NOT NULL,
    crawled_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,  -- 마지막으로 방문한 시각
    changed_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP   -- 내용이 마지막으로 바뀐 시각
)
""")
# 시각 컬럼을 둘로 나눈 것이 핵심이다.
# crawled_at은 "언제 확인했나"(신선도), changed_at은 "언제 바뀌었나"(개정 이력).
# 하나로 합치면 둘 중 하나를 영영 알 수 없다.


def save_with_change_detection(url, text):
    """내용이 바뀌었을 때만 changed_at을 갱신한다."""
    # sha256: 어떤 길이의 글이든 항상 64자짜리 지문으로 바꿔주는 함수.
    # 글자 하나만 달라져도 지문이 완전히 달라지기 때문에 변경 감지에 쓸 수 있다.
    new_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()

    row = conn2.execute("SELECT content_hash FROM crawled_documents WHERE url = ?", (url,)).fetchone()

    if row is None:
        conn2.execute(
            "INSERT INTO crawled_documents (url, text_content, content_hash) VALUES (?, ?, ?)",
            (url, text, new_hash),
        )
        result = "신규"
    elif row[0] == new_hash:
        # 내용이 같다 -> 방문 시각만 갱신하고 changed_at은 그대로 둔다
        conn2.execute("UPDATE crawled_documents SET crawled_at = CURRENT_TIMESTAMP WHERE url = ?", (url,))
        result = "변경 없음"
    else:
        conn2.execute(
            "UPDATE crawled_documents SET text_content = ?, content_hash = ?,"
            " crawled_at = CURRENT_TIMESTAMP, changed_at = CURRENT_TIMESTAMP WHERE url = ?",
            (text, new_hash, url),
        )
        result = "변경됨!"

    conn2.commit()
    print(f"  {result:<8} hash={new_hash[:12]}...")


print("1회차 크롤링:")
save_with_change_detection("https://ex.com/rule", "제9조 1주 40시간으로 한다.")
print("2회차 (내용 동일):")
save_with_change_detection("https://ex.com/rule", "제9조 1주 40시간으로 한다.")
print("3회차 (개정됨):")
save_with_change_detection("https://ex.com/rule", "제9조 1주 36시간으로 한다.")

**결과를 읽는 법**: 2회차에서 해시가 같아 "변경 없음"으로 넘어갔습니다.

이게 있으면 할 수 있는 일:
- **재색인 비용 절감** — 안 바뀐 문서는 임베딩을 다시 만들 필요가 없습니다. 임베딩은 유료입니다.
- **개정 알림** — `changed_at`이 어제인 문서만 뽑으면 "어제 개정된 규정" 목록이 됩니다.
- **디버깅** — "이 조항이 언제부터 이렇게 바뀌었나"를 추적할 수 있습니다.

> ⚠️ **주의**: 페이지에 현재 시각이나 조회수처럼 매번 바뀌는 요소가 있으면
> 내용이 그대로여도 해시가 달라집니다. 그래서 해시는 **정제된 본문**으로 계산해야 합니다.
> `extract_text_from_html()`을 거친 뒤에 해시를 뜨는 게 그래서 중요합니다.

## 정리

세 문제 모두 같은 이야기를 합니다.

| 연습 | 막아주는 실패 |
|---|---|
| Content-Type 판별 | PDF를 HTML로 착각해 깨진 글자를 저장 (조용한 실패) |
| 실패 기록 | 터미널을 닫으면 사라지는 정보 |
| 해시 변경 감지 | "언제 바뀌었나"를 영영 알 수 없게 되는 것 |

**나중에 알아채면 비싼 것을 지금 알아채게 만드는 것** — 이 시리즈 전체를 관통하는 주제입니다.

본문으로 돌아가기: [01_crawl_storage.ipynb](01_crawl_storage.ipynb)